In [ ]:
spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit
spark = (SparkSession.builder.appName("Jupyter") \
         .master("spark://spark-iceberg:7077") \
         .config("spark.driver.extraClassPath", "/opt/spark/jars/postgresql-42.6.0.jar") \
         .config("spark.executor.extraClassPath", "/opt/spark/jars/postgresql-42.6.0.jar") \
         .getOrCreate()
)

#spark = SparkSession.builder.config("spark.driver.extraClassPath", "/opt/spark/jars/postgresql-42.6.0.jar") \
#        .config("spark.executor.extraClassPath", "/opt/spark/jars/postgresql-42.6.0.jar") \
#        .appName("Jupyter").getOrCreate()
spark

24/12/07 19:02:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
try:
    df_postgres = spark.read.format("jdbc")\
            .option("url","jdbc:postgresql://host.docker.internal:5432/")\
            .option("driver","org.postgresql.Driver")\
            .option("user","postgres")\
            .option("password","postgres")\
            .option("dbtable","actor_films").load()
except Exception as e:
    print("An error occurred:", e)
    import traceback
    traceback.print_exc()

In [14]:
df_postgres.show()

+-------------+---------+--------------------+----+------+------+---------+
|        actor|  actorid|                film|year| votes|rating|   filmid|
+-------------+---------+--------------------+----+------+------+---------+
| Fred Astaire|nm0000001|         Ghost Story|1981|  7731|   6.3|tt0082449|
| Fred Astaire|nm0000001|     The Purple Taxi|1977|   533|   6.6|tt0076851|
| Fred Astaire|nm0000001|The Amazing Dober...|1976|   369|   5.3|tt0074130|
| Fred Astaire|nm0000001|The Towering Inferno|1974| 39888|   7.0|tt0072308|
|Lauren Bacall|nm0000002|  Ernest & Celestine|2012| 18793|   7.9|tt1816518|
|Lauren Bacall|nm0000002|          The Forger|2012|  4472|   5.4|tt1368858|
|Lauren Bacall|nm0000002|          All at Sea|2010|   110|   5.7|tt0858500|
|Lauren Bacall|nm0000002|          The Walker|2007|  5256|   5.8|tt0783608|
|Lauren Bacall|nm0000002|           Manderlay|2005| 22622|   7.3|tt0342735|
|Lauren Bacall|nm0000002|These Foolish Things|2005|   358|   5.6|tt0439848|
|Lauren Baca

In [ ]:
     .config("spark.executor.memory", "70g")
     .config("spark.driver.memory", "50g")
     .config("spark.memory.offHeap.enabled",True)
     .config("spark.memory.offHeap.size","64g") 

In [4]:
print(spark.conf.get('spark.driver.memory'))
print(spark.conf.get('spark.memory.offHeap.size'))

Py4JJavaError: An error occurred while calling o48.get.
: org.apache.spark.SparkNoSuchElementException: [SQL_CONF_NOT_FOUND] The SQL config "spark.driver.memory" cannot be found. Please verify that the config exists.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.sqlConfigNotFoundError(QueryExecutionErrors.scala:1984)
	at org.apache.spark.sql.internal.SQLConf.$anonfun$getConfString$3(SQLConf.scala:5274)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.internal.SQLConf.getConfString(SQLConf.scala:5274)
	at org.apache.spark.sql.RuntimeConfig.get(RuntimeConfig.scala:81)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(M

In [3]:
df = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/events.csv") \
            .withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))

df.show()

+-----------+----------+--------+--------------------+----------+--------------------+-------------------+
|    user_id| device_id|referrer|                host|       url|          event_time|         event_date|
+-----------+----------+--------+--------------------+----------+--------------------+-------------------+
| 1037710827| 532630305|    NULL| www.zachwilson.tech|         /|2021-03-08 17:27:...|2021-03-08 00:00:00|
|  925588856| 532630305|    NULL|    www.eczachly.com|         /|2021-05-10 11:26:...|2021-05-10 00:00:00|
|-1180485268| 532630305|    NULL|admin.zachwilson....|         /|2021-02-17 16:19:...|2021-02-17 00:00:00|
|-1044833855| 532630305|    NULL| www.zachwilson.tech|         /|2021-09-24 15:53:...|2021-09-24 00:00:00|
|  747494706| 532630305|    NULL| www.zachwilson.tech|         /|2021-09-26 16:03:...|2021-09-26 00:00:00|
|  747494706| 532630305|    NULL|admin.zachwilson....|         /|2021-02-21 16:08:...|2021-02-21 00:00:00|
| -824540328| 532630305|    NULL|admi

In [ ]:
spark.stop()

In [ ]:
df = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/events.csv") \
            .withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))

df.show()
# df.join(df, lit(1) == lit(1)).take(5)

In [ ]:
sorted = df.repartition(10, col("event_date")) \
        .sortWithinPartitions(col("event_date"), col("host")) \
        .withColumn("event_time", col("event_time").cast("timestamp")) \

 #, col("browser_family")) \
sorted.show()

sorted2 = df.repartition(10, col("event_date")) \
        .sort(col("event_date"), col("host")) \
        .withColumn("event_time", col("event_time").cast("timestamp")) \

 #, col("browser_family")) \
sorted2.show()

In [ ]:
%%sql

CREATE DATABASE IF NOT EXISTS bootcamp

In [ ]:
%%sql

DROP TABLE IF EXISTS bootcamp.events

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events (
    url STRING,
    referrer STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));

--browser_family STRING,/


In [ ]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_sorted (
    url STRING,
    referrer STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg;

--browser_family STRING,


In [ ]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_unsorted (
    url STRING,
    referrer STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg;

--    browser_family STRING,


In [ ]:

start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    

first_sort_df = start_df.sortWithinPartitions(col("event_date"), col("host")) #, col("browser_family"), col("host"))

sorted = df.repartition(10, col("event_date")) \
        .sortWithinPartitions(col("event_date")) \
        .withColumn("event_time", col("event_time").cast("timestamp")) \

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [ ]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files





In [ ]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM demo.bootcamp.events.files;

In [ ]:
%%sql 
SELECT COUNT(1) FROM bootcamp.matches_bucketed.files